In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from rapidfuzz import fuzz
import fuzzy
import jellyfish
from Levenshtein import distance as levenshtein_distance
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. ЗАВАНТАЖЕННЯ ТА ПІДГОТОВКА ДАНИХ
# ============================================================

print("=" * 70)
print("ЛАБОРАТОРНА РОБОТА №3: НЕЧІТКА ЛОГІКА")
print("Пошук подібних адрес у базі даних нерухомості Парижа")
print("=" * 70)

# Завантажуємо реальний датасет (вкажіть правильний шлях до файлу)
df_full = pd.read_csv('ads.csv')

# Беремо вибірку з 50-100 записів для тестування, щоб скрипт відпрацював швидко
df = df_full.head(50).copy() 

# Заповнюємо можливі порожні значення
df['description'] = df['description'].fillna('')
df['adress'] = df['adress'].fillna('') 

print(f"\n✓ Датасет завантажено. Для аналізу взято: {len(df)} записів")
print(f"  Колонки: {list(df.columns)}")

# ============================================================
# 2. ФУНКЦІЇ СХОЖОСТІ
# ============================================================

print("\n" + "=" * 70)
print("2. ФУНКЦІЇ ОБЧИСЛЕННЯ СХОЖОСТІ")
print("=" * 70)

soundex = fuzzy.Soundex(4)

def soundex_similarity(str1: str, str2: str) -> float:
    """Фонетична схожість через Soundex (0 або 1)"""
    try:
        # Беремо перше слово для Soundex
        w1 = str(str1).split()[0] if str(str1).split() else str1
        w2 = str(str2).split()[0] if str(str2).split() else str2
        return 1.0 if soundex(w1) == soundex(w2) else 0.0
    except Exception:
        return 0.0

def jaro_winkler_similarity(str1: str, str2: str) -> float:
    """Схожість рядків Jaro-Winkler (0..1)"""
    try:
        return jellyfish.jaro_winkler_similarity(str(str1).lower(), str(str2).lower())
    except Exception:
        return 0.0

def damerau_levenshtein_dist(str1: str, str2: str) -> int:
    """Відстань Дамерау-Левенштейна"""
    try:
        return levenshtein_distance(str(str1).lower(), str(str2).lower())
    except Exception:
        return 999

def token_sort_ratio(str1: str, str2: str) -> float:
    """Token Sort Ratio з RapidFuzz (0..1)"""
    try:
        return fuzz.token_sort_ratio(str(str1).lower(), str(str2).lower()) / 100.0
    except Exception:
        return 0.0

def normalize_dl_distance(dist: int, max_len: int) -> float:
    """Нормалізація відстані DL до [0,1], де 1 = ідентичні"""
    if max_len == 0:
        return 1.0
    return max(0.0, 1.0 - dist / max_len)

print("  ✓ Soundex (фонетична схожість)")
print("  ✓ Jaro-Winkler (схожість рядків)")
print("  ✓ Damerau-Levenshtein (редакційна відстань)")
print("  ✓ Token Sort Ratio (RapidFuzz)")


# ============================================================
# 3. НЕЧІТКІ МНОЖИНИ ТА ФУНКЦІЇ НАЛЕЖНОСТІ
# ============================================================

print("\n" + "=" * 70)
print("3. НЕЧІТКІ МНОЖИНИ ТА ФУНКЦІЇ НАЛЕЖНОСТІ")
print("=" * 70)

def trimf(x: float, a: float, b: float, c: float) -> float:
    """Трикутна функція належності"""
    if x <= a or x >= c:
        return 0.0
    elif x <= b:
        return (x - a) / (b - a) if b != a else 1.0
    else:
        return (c - x) / (c - b) if c != b else 1.0

def trapmf(x: float, a: float, b: float, c: float, d: float) -> float:
    """Трапецієподібна функція належності"""
    if x <= a or x >= d:
        return 0.0
    elif x <= b:
        return (x - a) / (b - a) if b != a else 1.0
    elif x <= c:
        return 1.0
    else:
        return (d - x) / (d - c) if d != c else 1.0

# --- Ступінь схожості JW (0..1) ---
def mu_jw_low(x):    return trapmf(x, 0.0, 0.0, 0.3, 0.5)
def mu_jw_medium(x): return trimf(x, 0.3, 0.6, 0.9)
def mu_jw_high(x):   return trapmf(x, 0.7, 0.85, 1.0, 1.0)

# --- Фонетична схожість Soundex (0 або 1) ---
def mu_phon_no(x):      return trapmf(x, 0.0, 0.0, 0.1, 0.3)
def mu_phon_partial(x): return trimf(x, 0.2, 0.5, 0.8)
def mu_phon_full(x):    return trapmf(x, 0.7, 0.9, 1.0, 1.0)

# --- Нормалізована відстань DL (0..1, де 1=ідентичні) ---
def mu_dl_low(x):    return trapmf(x, 0.0, 0.0, 0.3, 0.5)
def mu_dl_medium(x): return trimf(x, 0.3, 0.55, 0.8)
def mu_dl_high(x):   return trapmf(x, 0.65, 0.8, 1.0, 1.0)

# --- Довжина терміну (в символах) ---
def mu_len_short(x):    return trapmf(x, 0, 0, 30, 50)
def mu_len_medium(x):   return trimf(x, 40, 70, 100)
def mu_len_long(x):     return trapmf(x, 80, 120, 1000, 1000)

print("  Визначено нечіткі множини:")
print("  • Довжина:       [коротка | середня | довга]")
print("  • Схожість JW:   [низька | середня | висока]")
print("  • Фонетика:      [невідповідна | часткова | повна]")
print("  • Відстань DL:   [низька | середня | висока]")


# ============================================================
# 4. БАЗА ПРАВИЛ НЕЧІТКОГО ВИВЕДЕННЯ (МАМДАНІ)
# ============================================================

print("\n" + "=" * 70)
print("4. БАЗА ПРАВИЛ НЕЧІТКОГО ВИВЕДЕННЯ (МЕТОД МАМДАНІ)")
print("=" * 70)

rules_description = [
    "ЯКЩО JW_висока ТА Phon_повна ТА DL_висока → належать (ВИСОКА)",
    "ЯКЩО JW_висока ТА DL_висока                → належать (ВИСОКА)",
    "ЯКЩО JW_висока ТА Phon_часткова            → належать (СЕРЕДНЯ-ВИСОКА)",
    "ЯКЩО JW_середня ТА Phon_повна              → належать (СЕРЕДНЯ)",
    "ЯКЩО JW_середня ТА DL_середня             → вірогідно (СЕРЕДНЯ)",
    "ЯКЩО JW_низька ТА Phon_невідповідна        → не належать (НИЗЬКА)",
    "ЯКЩО JW_низька ТА DL_низька               → не належать (НИЗЬКА)",
    "ЯКЩО JW_середня ТА Phon_невідповідна      → мало вірогідно (НИЗЬКА-СЕРЕДНЯ)",
]

for i, rule in enumerate(rules_description, 1):
    print(f"  Правило {i}: {rule}")

def apply_fuzzy_rules(jw: float, phon: float, dl_norm: float, term_len: float) -> float:
    """
    Нечітке виведення методом Мамдані.
    Повертає значення виходу [0..1] (ймовірність належності).
    """
    # Ступені активації правил (існуючі)
    r1 = min(mu_jw_high(jw), mu_phon_full(phon), mu_dl_high(dl_norm))  
    r2 = min(mu_jw_high(jw), mu_dl_high(dl_norm))                      
    r3 = min(mu_jw_high(jw), mu_phon_partial(phon))                    
    r4 = min(mu_jw_medium(jw), mu_phon_full(phon))                     
    r5 = min(mu_jw_medium(jw), mu_dl_medium(dl_norm))                  
    r6 = min(mu_jw_low(jw), mu_phon_no(phon))                          
    r7 = min(mu_jw_low(jw), mu_dl_low(dl_norm))                        
    r8 = min(mu_jw_medium(jw), mu_phon_no(phon))                       
    
    # НОВЕ ПРАВИЛО з методички: "Якщо ступінь схожості середня та довжина термінів коротка, то ймовірність належності висока"
    r9 = min(mu_jw_medium(jw), mu_len_short(term_len))

    # Центроїдний метод дефазифікації (зважена сума)
    output_centers = {
        r1: 0.95,
        r2: 0.90,
        r3: 0.80,
        r4: 0.65,
        r5: 0.55,
        r6: 0.10,
        r7: 0.10,
        r8: 0.30,
        r9: 0.85, # Висока ймовірність для нового правила
    }
    
    numerator = sum(strength * center for strength, center in output_centers.items())
    denominator = sum(output_centers.keys())
    
    if denominator == 0:
        return 0.0
    return round(numerator / denominator, 4)


# ============================================================
# 5. ДЕФАЗИФІКАЦІЯ ТА ПРИЙНЯТТЯ РІШЕННЯ
# ============================================================

print("\n" + "=" * 70)
print("5. ОБЧИСЛЕННЯ СХОЖОСТІ ТА ДЕФАЗИФІКАЦІЯ")
print("=" * 70)

THRESHOLD = 0.70  # Поріг належності 70%

def compare_records(row1: pd.Series, row2: pd.Series) -> dict:
    """Порівнює два записи і повертає метрики схожості."""
    desc1, desc2 = str(row1['description']), str(row2['description'])
    addr1, addr2 = str(row1['adress']), str(row2['adress']) # Змінено на adress
    
    # JW для опису та адреси
    jw_desc = jaro_winkler_similarity(desc1, desc2)
    jw_addr = jaro_winkler_similarity(addr1, addr2)
    jw_combined = (jw_desc * 0.35 + jw_addr * 0.65)  
    
    # Soundex для першого слова адреси
    phon_addr = soundex_similarity(addr1, addr2)
    phon_desc = soundex_similarity(desc1, desc2)
    phon_combined = (phon_desc * 0.3 + phon_addr * 0.7)
    
    # DL для адреси (нормалізована)
    dl_dist = damerau_levenshtein_dist(addr1, addr2)
    max_len = max(len(addr1), len(addr2), 1)
    dl_norm = normalize_dl_distance(dl_dist, max_len)
    
    # Середня довжина терміну (адреси) для нового правила
    avg_len = (len(addr1) + len(addr2)) / 2
    
    # Token sort ratio
    tsr_desc = token_sort_ratio(desc1, desc2)
    tsr_addr = token_sort_ratio(addr1, addr2)
    
    # Нечітке виведення (додано avg_len)
    fuzzy_score = apply_fuzzy_rules(jw_combined, phon_combined, dl_norm, avg_len)
    
    # Підсилення через token sort ratio (хедж "дуже схожі токени")
    if tsr_addr > 0.85:
        fuzzy_score = min(1.0, fuzzy_score * 1.15)
    
    probability_pct = round(fuzzy_score * 100, 1)
    decision = "✅ Належить" if fuzzy_score >= THRESHOLD else "❌ Не належить"
    
    return {
        'id1': row1['id'],
        'id2': row2['id'],
        'desc1': desc1[:40] + '...' if len(desc1) > 40 else desc1,
        'desc2': desc2[:40] + '...' if len(desc2) > 40 else desc2,
        'addr1': addr1,
        'addr2': addr2,
        'jw_desc': round(jw_desc, 3),
        'jw_addr': round(jw_addr, 3),
        'jw_combined': round(jw_combined, 3),
        'soundex': round(phon_combined, 2),
        'dl_dist': dl_dist,
        'dl_norm': round(dl_norm, 3),
        'tsr_addr': round(tsr_addr, 3),
        'fuzzy_score': round(fuzzy_score, 3),
        'probability': probability_pct,
        'decision': decision,
    }


# ============================================================
# 6. ПОБУДОВА ТАБЛИЦІ ПОРІВНЯНЬ
# ============================================================

print(f"\n  Поріг належності: {THRESHOLD*100:.0f}%")
print("  Виконується порівняння всіх пар записів...")

results = []
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        result = compare_records(df.iloc[i], df.iloc[j])
        results.append(result)

results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values('probability', ascending=False)

# Топ-10 результатів
top10 = results_df_sorted.head(10)

print(f"\n  Всього порівнянь: {len(results)}")
print(f"  Виявлено схожих пар (≥{THRESHOLD*100:.0f}%): {len(results_df[results_df['fuzzy_score'] >= THRESHOLD])}")

# Таблиця топ-10
print("\n" + "=" * 70)
print("6. ТОП-10 НАЙБІЛЬШ СХОЖИХ ПАРНИХ ЗАПИСІВ")
print("=" * 70)

display_cols = ['id1', 'id2', 'addr1', 'addr2', 
                'jw_addr', 'soundex', 'dl_dist', 'probability', 'decision']
display_headers = ['ID1', 'ID2', 'Адреса 1', 'Адреса 2', 
                   'JW', 'Soundex', 'DL', 'Ймовірність', 'Рішення']

top10_display = top10[display_cols].copy()
top10_display['addr1'] = top10_display['addr1'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)
top10_display['addr2'] = top10_display['addr2'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)
top10_display['probability'] = top10_display['probability'].apply(lambda x: f"{x}%")

print(tabulate(top10_display, headers=display_headers, 
               tablefmt='grid', showindex=False))


# ============================================================
# 7. ДЕТАЛЬНИЙ ВИВІД СИСТЕМИ
# ============================================================

print("\n" + "=" * 70)
print("7. ДЕТАЛЬНИЙ ВИВІД СИСТЕМИ")
print("=" * 70)

detail_headers = [
    "Термін 1 (Адреса)", "Термін 2 (Адреса)",
    "Jaro-Winkler", "Soundex", "Damerau-Lev.", "Ймовірність", "Рішення"
]
detail_rows = []
for _, row in top10.iterrows():
    a1 = row['addr1'][:28] + '..' if len(row['addr1']) > 28 else row['addr1']
    a2 = row['addr2'][:28] + '..' if len(row['addr2']) > 28 else row['addr2']
    detail_rows.append([
        a1, a2,
        f"{row['jw_addr']:.2f}",
        f"{row['soundex']:.2f}",
        str(row['dl_dist']),
        f"{row['probability']}%",
        row['decision']
    ])

print(tabulate(detail_rows, headers=detail_headers, tablefmt='grid'))


# ============================================================
# 8. ВІЗУАЛІЗАЦІЯ ФУНКЦІЙ НАЛЕЖНОСТІ
# ============================================================

print("\n" + "=" * 70)
print("8. ПОБУДОВА ГРАФІКІВ ФУНКЦІЙ НАЛЕЖНОСТІ")
print("=" * 70)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Функції належності нечітких множин\nЛабораторна робота №3: Нечітка логіка', 
             fontsize=13, fontweight='bold', y=1.02)

x = np.linspace(0, 1, 500)

# --- Графік 1: Схожість Jaro-Winkler ---
ax = axes[0]
ax.plot(x, [mu_jw_low(xi) for xi in x], 'b-', linewidth=2, label='Низька')
ax.plot(x, [mu_jw_medium(xi) for xi in x], 'g-', linewidth=2, label='Середня')
ax.plot(x, [mu_jw_high(xi) for xi in x], 'r-', linewidth=2, label='Висока')
ax.set_title('Схожість Jaro-Winkler', fontsize=11, fontweight='bold')
ax.set_xlabel('Ступінь схожості JW', fontsize=10)
ax.set_ylabel('Ступінь належності μ(x)', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.1)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.4, label='_nolegend_')
ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.4)

# --- Графік 2: Фонетична схожість Soundex ---
ax = axes[1]
ax.plot(x, [mu_phon_no(xi) for xi in x], 'b-', linewidth=2, label='Невідповідна')
ax.plot(x, [mu_phon_partial(xi) for xi in x], 'g-', linewidth=2, label='Часткова')
ax.plot(x, [mu_phon_full(xi) for xi in x], 'r-', linewidth=2, label='Повна')
ax.set_title('Фонетична схожість (Soundex)', fontsize=11, fontweight='bold')
ax.set_xlabel('Ступінь фонетичної схожості', fontsize=10)
ax.set_ylabel('Ступінь належності μ(x)', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.1)

# --- Графік 3: Нормалізована відстань DL ---
ax = axes[2]
ax.plot(x, [mu_dl_low(xi) for xi in x], 'b-', linewidth=2, label='Низька схожість')
ax.plot(x, [mu_dl_medium(xi) for xi in x], 'g-', linewidth=2, label='Середня схожість')
ax.plot(x, [mu_dl_high(xi) for xi in x], 'r-', linewidth=2, label='Висока схожість')
ax.set_title('Схожість Damerau-Levenshtein\n(нормалізована)', fontsize=11, fontweight='bold')
ax.set_xlabel('Нормалізована схожість (1 − dist/len)', fontsize=10)
ax.set_ylabel('Ступінь належності μ(x)', fontsize=10)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.savefig('membership_functions.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ Збережено: membership_functions.png")


# ============================================================
# 9. ВІЗУАЛІЗАЦІЯ РОЗПОДІЛУ ЙМОВІРНОСТЕЙ
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Аналіз результатів нечіткого зіставлення адрес\nЛабораторна робота №3', 
             fontsize=13, fontweight='bold')

# Гістограма
ax = axes[0]
probs = results_df['probability'].values
colors = ['#2ecc71' if p >= 70 else '#e74c3c' for p in probs]
ax.hist(probs, bins=20, color='#3498db', edgecolor='white', alpha=0.8)
ax.axvline(x=THRESHOLD*100, color='red', linewidth=2, linestyle='--', 
           label=f'Поріг = {THRESHOLD*100:.0f}%')
ax.set_title('Розподіл ймовірностей належності', fontsize=11, fontweight='bold')
ax.set_xlabel('Ймовірність належності (%)', fontsize=10)
ax.set_ylabel('Кількість пар', fontsize=10)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Scatter: JW vs DL
ax = axes[1]
belong = results_df[results_df['fuzzy_score'] >= THRESHOLD]
not_belong = results_df[results_df['fuzzy_score'] < THRESHOLD]
ax.scatter(not_belong['jw_addr'], not_belong['dl_norm'], 
           c='#e74c3c', alpha=0.6, s=40, label='Не належить', zorder=3)
ax.scatter(belong['jw_addr'], belong['dl_norm'], 
           c='#2ecc71', alpha=0.8, s=60, label='Належить', zorder=4)
ax.set_title('Схожість JW vs DL-норм (за рішенням)', fontsize=11, fontweight='bold')
ax.set_xlabel('Jaro-Winkler (адреса)', fontsize=10)
ax.set_ylabel('DL нормалізована (адреса)', fontsize=10)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.axvline(x=0.7, color='gray', linestyle=':', alpha=0.5)
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('similarity_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ Збережено: similarity_analysis.png")


# ============================================================
# 10. ГРУП УВАННЯ СХОЖИХ ЗАПИСІВ
# ============================================================

print("\n" + "=" * 70)
print("9. ГРУПУВАННЯ СХОЖИХ ЗАПИСІВ")
print("=" * 70)

# Union-Find для групування
parent = {i: i for i in range(1, len(df) + 1)}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    parent[find(x)] = find(y)

# Об'єднати записи, що "належать"
for _, row in results_df[results_df['fuzzy_score'] >= THRESHOLD].iterrows():
    union(int(row['id1']), int(row['id2']))

# Групи
from collections import defaultdict
groups = defaultdict(list)
for record_id in df['id']:
    groups[find(record_id)].append(record_id)

groups_list = [(root, ids) for root, ids in groups.items() if len(ids) > 1]
groups_list.sort(key=lambda x: -len(x[1]))

print(f"\n  Виявлено груп схожих записів: {len(groups_list)}")
for root, ids in groups_list[:5]:
    print(f"\n  Група (коренева ID={root}):")
    for rid in ids:
        row = df[df['id'] == rid].iloc[0]
        addr = row['adress']
        addr_short = addr[:55] + '...' if len(addr) > 55 else addr
        print(f"    [{rid:2d}] {addr_short}")


# ============================================================
# 11. СТАТИСТИКА
# ============================================================

print("\n" + "=" * 70)
print("10. СТАТИСТИКА РЕЗУЛЬТАТІВ")
print("=" * 70)

total = len(results_df)
matched = len(results_df[results_df['fuzzy_score'] >= THRESHOLD])
not_matched = total - matched

print(f"\n  Загальна кількість порівнянь: {total}")
print(f"  Пар «належить» (≥{THRESHOLD*100:.0f}%):     {matched} ({matched/total*100:.1f}%)")
print(f"  Пар «не належить» (<{THRESHOLD*100:.0f}%):  {not_matched} ({not_matched/total*100:.1f}%)")
print(f"\n  Середня ймовірність (всі пари):        {results_df['probability'].mean():.1f}%")
print(f"  Середня ймовірність (схожі пари):      {results_df[results_df['fuzzy_score']>=THRESHOLD]['probability'].mean():.1f}%")
print(f"\n  Унікальних груп схожих записів:        {len(groups_list)}")
print(f"  Записів охоплено групами:               {sum(len(ids) for _, ids in groups_list)}")

ЛАБОРАТОРНА РОБОТА №3: НЕЧІТКА ЛОГІКА
Пошук подібних адрес у базі даних нерухомості Парижа

✓ Датасет завантажено. Для аналізу взято: 50 записів
  Колонки: ['id', 'description', 'adress', 'date_pub', 'date_update', 'price', 'fee']

2. ФУНКЦІЇ ОБЧИСЛЕННЯ СХОЖОСТІ
  ✓ Soundex (фонетична схожість)
  ✓ Jaro-Winkler (схожість рядків)
  ✓ Damerau-Levenshtein (редакційна відстань)
  ✓ Token Sort Ratio (RapidFuzz)

3. НЕЧІТКІ МНОЖИНИ ТА ФУНКЦІЇ НАЛЕЖНОСТІ
  Визначено нечіткі множини:
  • Довжина:       [коротка | середня | довга]
  • Схожість JW:   [низька | середня | висока]
  • Фонетика:      [невідповідна | часткова | повна]
  • Відстань DL:   [низька | середня | висока]

4. БАЗА ПРАВИЛ НЕЧІТКОГО ВИВЕДЕННЯ (МЕТОД МАМДАНІ)
  Правило 1: ЯКЩО JW_висока ТА Phon_повна ТА DL_висока → належать (ВИСОКА)
  Правило 2: ЯКЩО JW_висока ТА DL_висока                → належать (ВИСОКА)
  Правило 3: ЯКЩО JW_висока ТА Phon_часткова            → належать (СЕРЕДНЯ-ВИСОКА)
  Правило 4: ЯКЩО JW_середня ТА Phon_п

ValueError: invalid literal for int() with base 10: 'ad_1917'